# Hospital Receptionist #

In [21]:
import os
from dotenv import load_dotenv 
from openai import OpenAI 
import json 
import gradio as gr
import datetime as dt
import pytz

In [22]:
load_dotenv(override=True)
api=os.getenv('GOOGLE_API_KEY')
if api:
    print("api key is in my hands oh sorry its in my notebook hahahahahahahha")

api key is in my hands oh sorry its in my notebook hahahahahahahha


In [23]:
system_prompt="You are a **helpful, polite, and courteous Hospital Reception Assistant**. Your primary role is to guide patients efficiently to the correct counter or room for their appointments.\
**Core Responsibilities & Tool Integration:**\
1.  **Patient Guidance:** You will be provided with a **dictionary/data structure (the 'Tool')** containing doctor information (name, specialty, room/counter, and appointment cost). Use this tool to efficiently look up and communicate the following details to the patient:\
* The **Doctor's Name** and **Specialty**.\
* The appropriate **Counter or Room** the patient should head to.\
* The **Cost** of the appointment.\
2.  **Medical Inquiry Guardrail (CRITICAL):**\
* If a patient inquires about their specific medical condition, symptoms, or needs an assessment, **you must not provide a diagnosis, medical advice, or personal health assessment** under any circumstances, even if you know the answer internally.\
* Acknowledge their concern with politeness.\
* Your response **must be a firm, yet gentle referral**. State clearly that you are not a medical professional and must direct them to consult with a doctor specializing in the relevant area.\
**Tone:** Maintain a consistently **polite, helpful, and courteous** demeanor throughout the interaction."



ai=OpenAI(
    api_key=api,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)



In [24]:

def getdocinfo(doc_name):
    hospital_list = {
        "anjali sharma": {
            "specialty": "cardiology",
            "counter_room": "room 302 (cardio clinic)",
            "appointment_cost": "₹1,200",
            "timings": "09:00 AM - 01:00 PM"
        },
        "vikram singh": {
            "specialty": "orthopedics",
            "counter_room": "counter a-4 (bone & joint wing)",
            "appointment_cost": "₹950",
            "timings": "10:00 AM - 04:00 PM"
        },
        "leena kapoor": {
            "specialty": "pediatrics",
            "counter_room": "room 105 (children's health)",
            "appointment_cost": "₹800",
            "timings": "04:00 PM - 08:00 PM"
        },
        "rahul menon": {
            "specialty": "general medicine",
            "counter_room": "counter b-1 (internal medicine)",
            "appointment_cost": "₹750",
            "timings": "09:00 AM - 05:00 PM"
        },
        "zoya khan": {
            "specialty": "dermatology",
            "counter_room": "room 210 (skin clinic)",
            "appointment_cost": "₹1,100",
            "timings": "11:00 AM - 03:00 PM"
        },
        "sanjay rao": {
            "specialty": "neurology",
            "counter_room": "room 401 (neuro sciences)",
            "appointment_cost": "₹1,500",
            "timings": "02:00 PM - 06:00 PM"
        },
        "priya verma": {
            "specialty": "ophthalmology (eye specialist)",
            "counter_room": "room 205 (vision center)",
            "appointment_cost": "₹900",
            "timings": "10:00 AM - 02:00 PM"
        },
        "ahmed hassan": {
            "specialty": "gastroenterology (digestive system)",
            "counter_room": "counter c-3 (gi clinic)",
            "appointment_cost": "₹1,350",
            "timings": "05:00 PM - 09:00 PM"
        },
        "nisha reddy": {
            "specialty": "urology (urinary tract)",
            "counter_room": "room 315 (urology services)",
            "appointment_cost": "₹1,150",
            "timings": "08:00 AM - 12:00 PM"
        },
        "suresh jain": {
            "specialty": "pulmonology (lungs/respiratory)",
            "counter_room": "counter a-1 (respiratory care)",
            "appointment_cost": "₹1,050",
            "timings": "01:00 PM - 05:00 PM"
        }
    }
    
    print("doctor name is recived providing his information.")
    doc = doc_name.lower().replace("dr. ", "").strip()
    info = hospital_list.get(doc, "unknown")
    
    available = False
    
    if info != "unknown":
        time_range = info.get("timings", "")
        
        if time_range:
            
            def convert_to_24h_float(time_str):
                parts = time_str.strip().split()
                time_val = float(parts[0].replace(":", "."))
                period = parts[1]
                
                if period == "PM" and time_val < 12.0:
                    return time_val + 12.0
                elif period == "AM" and time_val >= 12.0:
                    return time_val - 12.0
                return time_val
                
            time_parts = time_range.split("-")
            
            start = convert_to_24h_float(time_parts[0])
            end = convert_to_24h_float(time_parts[1])
            
            try:
                tz = pytz.timezone("Asia/Kolkata")
                current_time_object = dt.datetime.now(tz)
                
                current_time_str = current_time_object.strftime("%H:%M").replace(":", ".")
                curr = float(current_time_str)
                
                available = curr < end and curr >= start
            
            except Exception as e:
                print(f"Error checking current time: {e}")
                
        return info | {"available": available}
        
    return info

In [25]:
getdocinfo("dr. ahmed hassan")

doctor name is recived providing his information.


{'specialty': 'gastroenterology (digestive system)',
 'counter_room': 'counter c-3 (gi clinic)',
 'appointment_cost': '₹1,350',
 'timings': '05:00 PM - 09:00 PM',
 'available': False}

In [26]:

def getprobleminfo(doc_name):
    specialty_list = {
        "skin": {
            "doctor": "zoya khan",
            "specialty": "dermatology",
            "timings": "11:00 AM - 03:00 PM"
        },
        "hair": {
            "doctor": "zoya khan",
            "specialty": "dermatology",
            "timings": "11:00 AM - 03:00 PM"
        },
        "nails": {
            "doctor": "zoya khan",
            "specialty": "dermatology",
            "timings": "11:00 AM - 03:00 PM"
        },
        "lungs": {
            "doctor": "suresh jain",
            "specialty": "pulmonology",
            "timings": "01:00 PM - 05:00 PM"
        },
        "respiratory_tract": {
            "doctor": "suresh jain",
            "specialty": "pulmonology",
            "timings": "01:00 PM - 05:00 PM"
        },
        "heart": {
            "doctor": "anjali sharma",
            "specialty": "cardiology",
            "timings": "09:00 AM - 01:00 PM"
        },
        "blood_vessels": {
            "doctor": "anjali sharma",
            "specialty": "cardiology",
            "timings": "09:00 AM - 01:00 PM"
        },
        "bones": {
            "doctor": "vikram singh",
            "specialty": "orthopedics",
            "timings": "10:00 AM - 04:00 PM"
        },
        "joints": {
            "doctor": "vikram singh",
            "specialty": "orthopedics",
            "timings": "10:00 AM - 04:00 PM"
        },
        "ligaments": {
            "doctor": "vikram singh",
            "specialty": "orthopedics",
            "timings": "10:00 AM - 04:00 PM"
        },
        "tendons": {
            "doctor": "vikram singh",
            "specialty": "orthopedics",
            "timings": "10:00 AM - 04:00 PM"
        },
        "brain": {
            "doctor": "sanjay rao",
            "specialty": "neurology",
            "timings": "02:00 PM - 06:00 PM"
        },
        "nerves": {
            "doctor": "sanjay rao",
            "specialty": "neurology",
            "timings": "02:00 PM - 06:00 PM"
        },
        "spinal_cord": {
            "doctor": "sanjay rao",
            "specialty": "neurology",
            "timings": "02:00 PM - 06:00 PM"
        },
        "stomach": {
            "doctor": "ahmed hassan",
            "specialty": "gastroenterology",
            "timings": "05:00 PM - 09:00 PM"
        },
        "liver": {
            "doctor": "ahmed hassan",
            "specialty": "gastroenterology",
            "timings": "05:00 PM - 09:00 PM"
        },
        "intestines": {
            "doctor": "ahmed hassan",
            "specialty": "gastroenterology",
            "timings": "05:00 PM - 09:00 PM"
        },
        "pancreas": {
            "doctor": "ahmed hassan",
            "specialty": "gastroenterology",
            "timings": "05:00 PM - 09:00 PM"
        },
        "esophagus": {
            "doctor": "ahmed hassan",
            "specialty": "gastroenterology",
            "timings": "05:00 PM - 09:00 PM"
        },
        "eyes": {
            "doctor": "priya verma",
            "specialty": "ophthalmology",
            "timings": "10:00 AM - 02:00 PM"
        },
        "kidneys": {
            "doctor": "nisha reddy",
            "specialty": "urology",
            "timings": "08:00 AM - 12:00 PM"
        },
        "urinary_tract": {
            "doctor": "nisha reddy",
            "specialty": "urology",
            "timings": "08:00 AM - 12:00 PM"
        },
        "children": {
            "doctor": "leena kapoor",
            "specialty": "pediatrics",
            "timings": "04:00 PM - 08:00 PM"
        },
        "infants": {
            "doctor": "leena kapoor",
            "specialty": "pediatrics",
            "timings": "04:00 PM - 08:00 PM"
        },
        "adults": {
            "doctor": "rahul menon",
            "specialty": "general medicine",
            "timings": "09:00 AM - 05:00 PM"
        }
    }
    
    print("problem name is recived providing information.")
    doc = doc_name.lower().replace("dr. ", "").strip()
    info = specialty_list.get(doc, "unknown")
    
    time = "" 
    available = False
    
    if info != "unknown":
        time = info.get("timings","").split("-")
        
        start=time[0].strip().split()
        if start[1]=="PM":
            start=float(start[0].replace(":","."))+12
        else:
            start=float(start[0].replace(":","."))
            
        end=time[1].strip().split()
        if end[1]=="PM":
            end=float(end[0].replace(":","."))+12
        else:
            end=float(end[0].replace(":","."))
        
        if time != "":
            tz=pytz.timezone("Asia/Kolkata")
            current=dt.datetime.now(tz).strftime("%H:%M").replace(":",".")
            curr=float(current)
            
            available=curr<end and curr >start
            
        return info | {"available":available}
        
    return info

In [27]:
getprobleminfo("intestines")

problem name is recived providing information.


{'doctor': 'ahmed hassan',
 'specialty': 'gastroenterology',
 'timings': '05:00 PM - 09:00 PM',
 'available': False}

In [28]:
Doctor_Function={
  "name": "getdocinfo",
  "description": "Takes doctor's name and checks their availability and returns the assigned doctor, specialty, and a real-time availability status (True/False) based on the current time and the doctor's office hours. Assumes current timezone is Asia/Kolkata (IST).",
  "parameters": {
    "type": "object",
    "properties": {
      "doc_name": {
        "type": "string",
        "description": "Tells the doctor's name which is to be found."
      }
    },
    "required": [
      "doc_name"
    ],
    "additionalProperties": False
  }
}

In [29]:
Problem_Function={
  "name": "getprobleminfo",
  "description": "Takes a medical problem or body part (e.g., 'skin', 'lungs', 'bones') as input, checks the specialist directory, and returns the assigned doctor, specialty, and a real-time availability status (True/False) based on the current time and the doctor's office hours. Assumes current timezone is Asia/Kolkata (IST).",
  "parameters": {
    "type": "object",
    "properties": {
      "doc_name": {
        "type": "string",
        "description": "The specific medical problem, organ, or body system (e.g., 'lungs', 'eyes', 'stomach', 'infants') to look up."
      }
    },
    "required": [
      "doc_name"
    ],
    "additionalProperties": False
  }
}

In [30]:
tools=[{"type":"function","function":Problem_Function},{"type":"function","function":Doctor_Function}]

In [31]:
import json

def handle_tool_call(message):
    toolcall = message.tool_calls[0]
    
    if toolcall.function.name == "getdocinfo":
        argu = json.loads(toolcall.function.arguments)
        print(argu)
        
        doc_name = argu.get("doc_name")
        print(doc_name)
        if doc_name is None:
            info = {"error": "Missing required doctor name argument."}
        else:
            
            info = getdocinfo(doc_name)
        
        
        response = {
            "role": "tool",
            "content": json.dumps(info),
            "tool_call_id": toolcall.id
        }
        
        
        return response, info

    
    elif toolcall.function.name == "getprobleminfo":
        argu = json.loads(toolcall.function.arguments)
        doc_name = argu.get("doc_name")
        
        if doc_name is None:
            info = {"error": "Missing required problem name argument."}
        else:
            
            info = getprobleminfo(doc_name)
            
        response = {
            "role": "tool",
            "content": json.dumps(info),
            "tool_call_id": toolcall.id
        }
        return response, info
        
    else:
        error_content = {"error": f"Tool {toolcall.function.name} is not recognized."}
        response = {
            "role": "tool",
            "content": json.dumps(error_content),
            "tool_call_id": toolcall.id
        }
        return response, None

In [32]:
def chat(message,history):
    messages=[{"role":"system","content":system_prompt}]+history+[{"role":"user","content":message}]
    response=ai.chat.completions.create(model="gemini-2.0-flash",messages=messages,tools=tools)


    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        response, _= handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        response = ai.chat.completions.create(model="gemini-2.0-flash", messages=messages)
    
    return response.choices[0].message.content

In [33]:
gr.ChatInterface(fn=chat,type="messages").launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


In [34]:
gr.ChatInterface(fn=chat,type="messages").launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


In [35]:
i want to meet dr. nisha reddy

SyntaxError: invalid syntax (3673601896.py, line 1)